# Notebook 11: Continuous Batching

**Series:** LLM Systems & Inference  
**Prerequisites:** Notebooks 05 (KV cache), 07 (PagedAttention), 10 (Speculative Decoding)

---

## Background: The Throughput Problem Nobody Talked About

By 2022, the LLM community had solved many *latency* problems. KV caching made single-request decode fast. Flash Attention enabled long contexts. Speculative decoding cut time-to-first-token. But there was a different problem lurking: **throughput** — how many requests can you serve *simultaneously*?

This matters enormously in production. A chatbot serving 10,000 users/day needs to handle bursts of concurrent requests efficiently. If each request ties up a GPU for its entire lifetime — from first input token to last output token — then the GPU spends most of its time idle, waiting for slow requests while fast ones have already finished.

### Static Batching: The Old Way

Traditional deep learning inference uses **static batching**: collect N requests, pad them all to the same length, run one batch, return results. This works beautifully for image classifiers — every image produces exactly one output, so batches are clean and uniform.

LLMs are completely different. Each request generates a *variable* number of output tokens:
- Request A: "What is 2+2?" → 3 tokens
- Request B: "Write me a short story about a robot" → 400 tokens
- Request C: "Translate this paragraph" → 120 tokens

With static batching, you must wait for the *longest* request to finish before freeing the batch. The GPU that finished Request A's 3 tokens at step 3 sits completely idle for the next 397 steps, waiting for Request B.

This is called **bubbling** — wasted compute cycles caused by length imbalance. In practice, bubbles account for 60–80% of wasted GPU time in naive LLM serving systems.

### The Orca Paper (2022)

In 2022, researchers from Seoul National University published ["Orca: A Distributed Serving System for Transformer-Based Generative Models"](https://www.usenix.org/conference/osdi22/presentation/yu). This paper introduced **continuous batching** (which they called "iteration-level scheduling"), and it transformed how the entire industry serves LLMs.

The insight: instead of treating a request as the atomic unit of scheduling, treat **each decode step** as the unit. After every token generation step, the scheduler can:
- **Evict** completed requests (those that hit EOS or max length)
- **Admit** new requests from the waiting queue
- **Preempt** low-priority requests to make room for new ones

This means the batch composition changes *at every step* — it's continuous, not static. A request that finishes early immediately frees its slot for a new arrival. The GPU is never waiting for stragglers.

### Why This Was Hard to Build

Continuous batching sounds simple, but it requires rethinking the entire serving stack:

1. **Variable-length batches**: The batch changes at every step — attention masks, position ids, and KV cache indices all change dynamically
2. **KV cache management**: Each request has a different KV cache size at each step — you can't pre-allocate a static 2D tensor for all requests
3. **Preemption**: If memory runs out mid-sequence, you need to either swap a request to CPU or recompute from scratch (recomputation is often faster)
4. **Scheduling policy**: Which requests to admit and which to hold matters enormously for fairness and latency percentiles

This is exactly why vLLM was built. It combines PagedAttention (notebook 07) with continuous batching to handle all of these challenges. Together, they produce **10–24× higher throughput** than Hugging Face's naive implementation for typical workloads.

---

## What You'll Build

1. **Static batching simulator** — the naive baseline with bubble analysis
2. **Continuous batching scheduler** — iteration-level admission/eviction
3. **Throughput comparison** — quantifying the speedup
4. **Preemption policies** — swap vs. recompute tradeoffs
5. **Latency distribution analysis** — what continuous batching does to tail latencies

In [ ]:
!pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass, field
from typing import Optional
from collections import deque
import random
import time

random.seed(42)
np.random.seed(42)

print("Setup complete.")

## Part 1: Request Model

Each LLM request has:
- An arrival time (Poisson process in production)
- A prompt length (input tokens)
- An output length (not known in advance — sampled from a distribution)
- A priority (for preemption policies)

In [ ]:
@dataclass
class Request:
    req_id: int
    arrival_step: int        # when this request arrived at the system
    prompt_len: int          # number of input tokens (prefill cost)
    output_len: int          # total output tokens to generate (ground truth, hidden from scheduler)
    priority: int = 0        # higher = higher priority
    
    # Runtime state (set by scheduler)
    start_step: Optional[int] = None    # when it started being served
    finish_step: Optional[int] = None   # when generation completed
    tokens_generated: int = 0
    preempt_count: int = 0
    
    @property
    def is_finished(self) -> bool:
        return self.tokens_generated >= self.output_len
    
    @property
    def wait_time(self) -> Optional[int]:
        if self.start_step is None:
            return None
        return self.start_step - self.arrival_step
    
    @property
    def total_time(self) -> Optional[int]:
        if self.finish_step is None:
            return None
        return self.finish_step - self.arrival_step
    
    @property
    def kv_cache_tokens(self) -> int:
        """Current KV cache size: prompt tokens + generated tokens."""
        return self.prompt_len + self.tokens_generated


def generate_requests(
    n_requests: int,
    arrival_rate: float = 0.3,   # avg requests per step (Poisson)
    prompt_len_range: tuple = (10, 200),
    output_len_range: tuple = (10, 500),
    seed: int = 42,
) -> list[Request]:
    """Generate a stream of requests with Poisson arrivals and variable lengths."""
    rng = np.random.RandomState(seed)
    requests = []
    current_step = 0
    
    for i in range(n_requests):
        # Poisson inter-arrival times
        inter_arrival = rng.geometric(p=arrival_rate)
        current_step += inter_arrival
        
        # Log-normal output lengths (realistic — some short, some very long)
        output_len = int(np.clip(
            rng.lognormal(mean=4.0, sigma=1.2),
            output_len_range[0], output_len_range[1]
        ))
        prompt_len = int(rng.uniform(*prompt_len_range))
        
        requests.append(Request(
            req_id=i,
            arrival_step=current_step,
            prompt_len=prompt_len,
            output_len=output_len,
        ))
    
    return requests


requests = generate_requests(200, arrival_rate=0.4)

output_lens = [r.output_len for r in requests]
print(f"Generated {len(requests)} requests")
print(f"Output length distribution:")
print(f"  Min: {min(output_lens)}, Max: {max(output_lens)}")
print(f"  Mean: {np.mean(output_lens):.0f}, Median: {np.median(output_lens):.0f}")
print(f"  P90: {np.percentile(output_lens, 90):.0f}, P99: {np.percentile(output_lens, 99):.0f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(output_lens, bins=30, color='#3498db', alpha=0.8, edgecolor='white')
axes[0].set_xlabel("Output Length (tokens)")
axes[0].set_ylabel("Count")
axes[0].set_title("Request Output Length Distribution (Log-Normal)")
axes[0].axvline(np.mean(output_lens), color='red', linestyle='--', label=f'Mean={np.mean(output_lens):.0f}')
axes[0].legend()

arrivals = [r.arrival_step for r in requests]
axes[1].plot(arrivals, range(len(arrivals)), color='#2ecc71')
axes[1].set_xlabel("Time Step")
axes[1].set_ylabel("Cumulative Requests")
axes[1].set_title("Request Arrival Process")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Part 2: Static Batching Simulator

The naive approach: collect a batch of requests, run them together until the **longest** one finishes, then start a new batch. Shorter requests block on longer ones.

In [ ]:
@dataclass
class StaticBatchStats:
    total_steps: int = 0
    total_tokens_generated: int = 0
    total_bubble_steps: int = 0   # steps where slots are idle
    n_batches: int = 0
    finished_requests: list = field(default_factory=list)
    
    @property
    def gpu_utilization(self) -> float:
        """Fraction of total GPU-slot-steps that were actually doing useful work."""
        return 1.0 - (self.total_bubble_steps / max(1, self.total_steps))
    
    @property
    def throughput(self) -> float:
        """Tokens generated per step (proxy for throughput)."""
        return self.total_tokens_generated / max(1, self.total_steps)


def simulate_static_batching(
    requests: list[Request],
    batch_size: int = 8,
    max_kv_tokens: int = 4096,   # total KV cache capacity in tokens
    prefill_cost: int = 1,        # steps for prefill (simplified: 1 step per batch)
) -> StaticBatchStats:
    """
    Static batching: wait for N requests, run together until longest finishes.
    
    Simulates the scheduling loop. Time is measured in 'steps' (decode iterations).
    Each step generates one token for each request in the current batch.
    """
    import copy
    reqs = [copy.copy(r) for r in requests]  # don't mutate originals
    stats = StaticBatchStats()
    
    queue = deque(sorted(reqs, key=lambda r: r.arrival_step))
    current_step = 0
    
    while queue:
        # Collect a batch of up to batch_size requests that have arrived
        batch = []
        temp_queue = deque()
        
        # Wait until we have enough requests
        while queue and len(batch) < batch_size:
            req = queue.popleft()
            if req.arrival_step <= current_step:
                batch.append(req)
            else:
                temp_queue.append(req)  # arrived later, put back
                break
        
        # Put non-arrived requests back
        while temp_queue:
            queue.appendleft(temp_queue.pop())
        
        if not batch:
            # No requests have arrived yet — skip to next arrival
            if queue:
                current_step = queue[0].arrival_step
            continue
        
        # Mark start
        for req in batch:
            req.start_step = current_step
        
        # Prefill step
        current_step += prefill_cost
        stats.total_steps += prefill_cost
        
        # Decode loop — run until ALL requests in batch are done
        max_output = max(req.output_len for req in batch)
        
        for step in range(max_output):
            # Count tokens actually generated this step (not idle slots)
            active = sum(1 for req in batch if not req.is_finished)
            idle = len(batch) - active
            
            stats.total_bubble_steps += idle  # wasted slots
            stats.total_tokens_generated += active
            stats.total_steps += 1
            current_step += 1
            
            for req in batch:
                if not req.is_finished:
                    req.tokens_generated += 1
                    if req.is_finished:
                        req.finish_step = current_step
        
        stats.n_batches += 1
        stats.finished_requests.extend(batch)
    
    return stats


static_stats = simulate_static_batching(requests, batch_size=8)

print("Static Batching Results:")
print(f"  Total steps:           {static_stats.total_steps:,}")
print(f"  Total tokens generated: {static_stats.total_tokens_generated:,}")
print(f"  Total bubble steps:    {static_stats.total_bubble_steps:,}")
print(f"  GPU utilization:       {static_stats.gpu_utilization:.1%}")
print(f"  Throughput:            {static_stats.throughput:.2f} tokens/step")
print(f"  Number of batches:     {static_stats.n_batches}")

wait_times = [r.wait_time for r in static_stats.finished_requests if r.wait_time is not None]
total_times = [r.total_time for r in static_stats.finished_requests if r.total_time is not None]
print(f"\nLatency:")
print(f"  Median wait time:      {np.median(wait_times):.0f} steps")
print(f"  P99 wait time:         {np.percentile(wait_times, 99):.0f} steps")
print(f"  Median total time:     {np.median(total_times):.0f} steps")

## Part 3: Continuous Batching Scheduler

The iteration-level scheduler: at every decode step, check if any requests have finished
and immediately replace them with waiting requests. The batch is never idle waiting for
slow requests.

In [ ]:
@dataclass
class ContinuousBatchStats:
    total_steps: int = 0
    total_tokens_generated: int = 0
    idle_steps: int = 0          # steps where batch was below max size (empty queue)
    preemptions: int = 0
    finished_requests: list = field(default_factory=list)
    batch_size_history: list = field(default_factory=list)
    
    @property
    def throughput(self) -> float:
        return self.total_tokens_generated / max(1, self.total_steps)


def simulate_continuous_batching(
    requests: list[Request],
    max_batch_size: int = 8,
    max_kv_tokens: int = 8192,   # total KV cache capacity (continuous batching uses it better)
    prefill_cost: int = 1,
) -> ContinuousBatchStats:
    """
    Continuous batching (iteration-level scheduling).
    
    At every step:
    1. Remove finished requests from active batch
    2. Admit new requests from queue (up to max_batch_size and memory limit)
    3. Run one decode step for all active requests
    """
    import copy
    reqs = [copy.copy(r) for r in requests]
    stats = ContinuousBatchStats()
    
    # Priority queue by arrival time
    waiting = sorted(reqs, key=lambda r: r.arrival_step)
    waiting_idx = 0
    active_batch: list[Request] = []
    current_step = 0
    
    while waiting_idx < len(waiting) or active_batch:
        # ── 1. Evict finished requests ──
        newly_finished = [r for r in active_batch if r.is_finished]
        for req in newly_finished:
            req.finish_step = current_step
            stats.finished_requests.append(req)
        active_batch = [r for r in active_batch if not r.is_finished]
        
        # ── 2. Admit new requests ──
        # Admit requests that have arrived and fit in batch/memory
        while (waiting_idx < len(waiting) and
               len(active_batch) < max_batch_size and
               waiting[waiting_idx].arrival_step <= current_step):
            
            # Check KV cache capacity
            current_kv = sum(r.kv_cache_tokens for r in active_batch)
            new_req = waiting[waiting_idx]
            if current_kv + new_req.prompt_len > max_kv_tokens:
                break  # Memory full — can't admit yet
            
            new_req.start_step = current_step
            active_batch.append(new_req)
            waiting_idx += 1
        
        # ── 3. If no active requests but queue has future arrivals, skip ahead ──
        if not active_batch:
            if waiting_idx < len(waiting):
                current_step = waiting[waiting_idx].arrival_step
                continue
            else:
                break
        
        # ── 4. Decode step ──
        batch_size_now = len(active_batch)
        stats.batch_size_history.append(batch_size_now)
        stats.total_tokens_generated += batch_size_now
        stats.total_steps += 1
        
        if batch_size_now < max_batch_size:
            stats.idle_steps += (max_batch_size - batch_size_now)
        
        for req in active_batch:
            req.tokens_generated += 1
        
        current_step += 1
    
    return stats


cont_stats = simulate_continuous_batching(requests, max_batch_size=8)

print("Continuous Batching Results:")
print(f"  Total steps:           {cont_stats.total_steps:,}")
print(f"  Total tokens generated: {cont_stats.total_tokens_generated:,}")
print(f"  Throughput:            {cont_stats.throughput:.2f} tokens/step")
print(f"  Avg batch size:        {np.mean(cont_stats.batch_size_history):.2f}")

wait_times_c = [r.wait_time for r in cont_stats.finished_requests if r.wait_time is not None]
total_times_c = [r.total_time for r in cont_stats.finished_requests if r.total_time is not None]
print(f"\nLatency:")
print(f"  Median wait time:      {np.median(wait_times_c):.0f} steps")
print(f"  P99 wait time:         {np.percentile(wait_times_c, 99):.0f} steps")
print(f"  Median total time:     {np.median(total_times_c):.0f} steps")

throughput_gain = cont_stats.throughput / static_stats.throughput
print(f"\nThroughput gain (continuous vs static): {throughput_gain:.2f}x")

## Part 4: Visualizing Batch Utilization

The most intuitive way to understand the difference: plot what the batch looks like over time.
Static batching has huge idle gaps; continuous batching keeps the batch full.

In [ ]:
def simulate_static_batching_with_history(
    requests: list[Request],
    batch_size: int = 4,
    n_steps_to_show: int = 200,
) -> list[list[Optional[int]]]:
    """Returns step-by-step batch occupancy for visualization."""
    import copy
    reqs = sorted([copy.copy(r) for r in requests], key=lambda r: r.arrival_step)
    history = []  # each entry: list of req_ids (None = idle slot)
    
    queue = deque(reqs)
    current_step = 0
    
    while queue and current_step < n_steps_to_show:
        # Fill batch
        batch = []
        while queue and len(batch) < batch_size:
            req = queue[0]
            if req.arrival_step <= current_step:
                batch.append(queue.popleft())
            else:
                break
        
        if not batch:
            current_step = queue[0].arrival_step if queue else current_step + 1
            continue
        
        max_out = max(r.output_len for r in batch)
        for req in batch:
            req.start_step = current_step
        
        for step_idx in range(max_out):
            if current_step >= n_steps_to_show:
                break
            slot_state = [
                req.req_id if step_idx < req.output_len else None
                for req in batch
            ] + [None] * (batch_size - len(batch))
            history.append(slot_state)
            current_step += 1
        
    # Pad to n_steps_to_show
    while len(history) < n_steps_to_show:
        history.append([None] * batch_size)
    return history[:n_steps_to_show]


def simulate_continuous_batching_with_history(
    requests: list[Request],
    batch_size: int = 4,
    n_steps_to_show: int = 200,
) -> list[list[Optional[int]]]:
    """Returns step-by-step batch occupancy for visualization."""
    import copy
    reqs = sorted([copy.copy(r) for r in requests], key=lambda r: r.arrival_step)
    history = []
    
    waiting = list(reqs)
    w_idx = 0
    active: list[Request] = []
    
    for current_step in range(n_steps_to_show):
        # Evict finished
        active = [r for r in active if not r.is_finished]
        
        # Admit new
        while w_idx < len(waiting) and len(active) < batch_size and waiting[w_idx].arrival_step <= current_step:
            waiting[w_idx].start_step = current_step
            active.append(waiting[w_idx])
            w_idx += 1
        
        # Record
        slot_state = [r.req_id for r in active] + [None] * (batch_size - len(active))
        history.append(slot_state)
        
        # Generate
        for r in active:
            r.tokens_generated += 1
    
    return history


N_SHOW = 150
BATCH_SIZE_VIZ = 6

static_hist = simulate_static_batching_with_history(requests[:30], BATCH_SIZE_VIZ, N_SHOW)
cont_hist = simulate_continuous_batching_with_history(requests[:30], BATCH_SIZE_VIZ, N_SHOW)

fig, axes = plt.subplots(2, 1, figsize=(16, 6))

# Assign colors per request ID
req_ids = sorted(set(v for row in static_hist + cont_hist for v in row if v is not None))
colors = plt.cm.tab20(np.linspace(0, 1, max(len(req_ids), 1)))
id_to_color = {rid: colors[i % len(colors)] for i, rid in enumerate(req_ids)}

for ax, hist, title in [
    (axes[0], static_hist, f"Static Batching (batch_size={BATCH_SIZE_VIZ})"),
    (axes[1], cont_hist, f"Continuous Batching (max_batch_size={BATCH_SIZE_VIZ})"),
]:
    util_matrix = np.zeros((BATCH_SIZE_VIZ, N_SHOW))
    color_matrix = np.ones((BATCH_SIZE_VIZ, N_SHOW, 4))  # RGBA
    
    for t, row in enumerate(hist):
        for slot, req_id in enumerate(row):
            if req_id is not None:
                util_matrix[slot, t] = 1
                color_matrix[slot, t] = [*id_to_color[req_id][:3], 0.85]
            else:
                color_matrix[slot, t] = [0.92, 0.92, 0.92, 1.0]  # light gray = idle
    
    ax.imshow(color_matrix, aspect='auto', origin='upper',
              extent=[0, N_SHOW, BATCH_SIZE_VIZ, 0])
    ax.set_yticks(range(BATCH_SIZE_VIZ))
    ax.set_yticklabels([f"Slot {i}" for i in range(BATCH_SIZE_VIZ)])
    ax.set_xlabel("Time Step")
    ax.set_title(title)
    
    utilization = np.mean(util_matrix)
    ax.text(0.98, 0.05, f"Utilization: {utilization:.0%}",
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=10, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Legend
legend_patches = [
    mpatches.Patch(color='#dedede', label='Idle (bubble)'),
    mpatches.Patch(color='#3498db', label='Active request'),
]
axes[0].legend(handles=legend_patches, loc='upper right', fontsize=8)

plt.suptitle("Batch Utilization: Static vs Continuous Batching", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('batching_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## Part 5: Throughput-Latency Tradeoff

Continuous batching also changes the latency distribution significantly.
Short requests no longer wait for long ones to finish before their results are returned.

In [ ]:
# Compare latency distributions
static_wait = [r.wait_time for r in static_stats.finished_requests if r.wait_time is not None]
cont_wait = [r.wait_time for r in cont_stats.finished_requests if r.wait_time is not None]

static_total = [r.total_time for r in static_stats.finished_requests if r.total_time is not None]
cont_total = [r.total_time for r in cont_stats.finished_requests if r.total_time is not None]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Wait time (time-to-first-token proxy)
ax = axes[0]
bins = np.linspace(0, max(max(static_wait), max(cont_wait)), 50)
ax.hist(static_wait, bins=bins, alpha=0.7, label='Static Batching', color='#e74c3c', density=True)
ax.hist(cont_wait, bins=bins, alpha=0.7, label='Continuous Batching', color='#2ecc71', density=True)
ax.set_xlabel("Wait Time (steps to first token)")
ax.set_ylabel("Density")
ax.set_title("Time-to-First-Token Distribution")
ax.legend()
ax.grid(alpha=0.3)

# Annotate P50 / P99
for times, color, label in [(static_wait, '#e74c3c', 'Static'), (cont_wait, '#2ecc71', 'Continuous')]:
    p50 = np.percentile(times, 50)
    p99 = np.percentile(times, 99)
    ax.axvline(p50, color=color, linestyle='--', alpha=0.8, linewidth=1.5)
    ax.axvline(p99, color=color, linestyle=':', alpha=0.8, linewidth=1.5)

# Total time
ax2 = axes[1]
bins2 = np.linspace(0, max(max(static_total), max(cont_total)), 50)
ax2.hist(static_total, bins=bins2, alpha=0.7, label='Static Batching', color='#e74c3c', density=True)
ax2.hist(cont_total, bins=bins2, alpha=0.7, label='Continuous Batching', color='#2ecc71', density=True)
ax2.set_xlabel("Total Time (steps, arrival to last token)")
ax2.set_ylabel("Density")
ax2.set_title("Total Request Latency Distribution")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('latency_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print("Latency comparison (steps):")
print(f"{'Metric':<25} {'Static':>10} {'Continuous':>12} {'Improvement':>13}")
print("-" * 65)
for label, s_times, c_times in [
    ("Wait P50 (TTFT)", static_wait, cont_wait),
    ("Wait P99 (TTFT)", static_wait, cont_wait),
    ("Total P50", static_total, cont_total),
    ("Total P99", static_total, cont_total),
]:
    pct = 50 if 'P50' in label else 99
    s_val = np.percentile(s_times, pct)
    c_val = np.percentile(c_times, pct)
    improvement = (s_val - c_val) / s_val
    print(f"{label:<25} {s_val:>10.0f} {c_val:>12.0f} {improvement:>12.1%} better")

## Part 6: The KV Cache Memory Pressure Problem

Continuous batching makes GPU compute more efficient, but it creates memory pressure.
With more requests in-flight simultaneously, each holding a growing KV cache, you can
easily run out of GPU memory.

When memory is exhausted, the scheduler must choose:
1. **Swap to CPU**: Move the KV cache of a preempted request to CPU RAM (fast resume, but PCIe is slow)
2. **Recompute**: Discard the KV cache and rerun the prefill when the request is re-admitted (wastes compute, but free in memory terms)

vLLM's initial design chose recomputation. Modern vLLM supports both.

In [ ]:
def simulate_memory_pressure(
    requests: list[Request],
    max_batch_size: int = 8,
    kv_bytes_per_token: float = 0.5,  # MB per token (e.g., 32 layers * 2 heads * 2 * 2B)
    total_kv_memory_mb: float = 4096, # 4 GB for KV cache
) -> dict:
    """Simulate KV memory usage over time during continuous batching."""
    import copy
    reqs = sorted([copy.copy(r) for r in requests], key=lambda r: r.arrival_step)
    
    max_kv_tokens = int(total_kv_memory_mb / kv_bytes_per_token)
    
    waiting = list(reqs)
    w_idx = 0
    active: list[Request] = []
    
    memory_history = []
    batch_size_history = []
    preemptions = 0
    
    for current_step in range(2000):
        if w_idx >= len(waiting) and not active:
            break
        
        # Evict finished
        active = [r for r in active if not r.is_finished]
        
        # Compute current KV usage
        current_kv_tokens = sum(r.kv_cache_tokens for r in active)
        
        # Admit new requests if memory allows
        while (w_idx < len(waiting) and 
               len(active) < max_batch_size and
               waiting[w_idx].arrival_step <= current_step):
            new_req = waiting[w_idx]
            if current_kv_tokens + new_req.prompt_len <= max_kv_tokens:
                new_req.start_step = current_step
                active.append(new_req)
                current_kv_tokens += new_req.prompt_len
                w_idx += 1
            else:
                break  # Memory exhausted
        
        # If over memory limit (shouldn't happen with admission control, but check)
        while current_kv_tokens > max_kv_tokens and active:
            # Preempt the request with the largest KV cache
            victim = max(active, key=lambda r: r.kv_cache_tokens)
            active.remove(victim)
            waiting.insert(w_idx, victim)  # Put back in queue
            current_kv_tokens -= victim.kv_cache_tokens
            victim.tokens_generated = 0  # Recompute from scratch
            preemptions += 1
        
        if not active:
            if w_idx < len(waiting):
                current_step = waiting[w_idx].arrival_step - 1
            continue
        
        memory_history.append(current_kv_tokens * kv_bytes_per_token)
        batch_size_history.append(len(active))
        
        for r in active:
            r.tokens_generated += 1
    
    return {
        'memory_history_mb': memory_history,
        'batch_size_history': batch_size_history,
        'preemptions': preemptions,
        'total_kv_memory_mb': total_kv_memory_mb,
        'max_kv_tokens': max_kv_tokens,
    }


mem_result = simulate_memory_pressure(requests, max_batch_size=8)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
mem_history = mem_result['memory_history_mb']
ax.plot(mem_history, color='#e74c3c', linewidth=0.8, alpha=0.9)
ax.axhline(mem_result['total_kv_memory_mb'], color='black', linestyle='--',
           label=f"Memory limit ({mem_result['total_kv_memory_mb']:.0f} MB)")
ax.fill_between(range(len(mem_history)), mem_history, alpha=0.2, color='#e74c3c')
ax.set_xlabel("Time Step")
ax.set_ylabel("KV Cache Memory (MB)")
ax.set_title("KV Cache Memory Usage Over Time")
ax.legend()
ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(mem_result['batch_size_history'], color='#3498db', linewidth=0.8, alpha=0.9)
ax2.axhline(8, color='black', linestyle='--', label='Max batch size')
ax2.set_xlabel("Time Step")
ax2.set_ylabel("Active Batch Size")
ax2.set_title("Batch Size Over Time")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('memory_pressure.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Memory utilization: {np.mean(mem_result['memory_history_mb']) / mem_result['total_kv_memory_mb']:.1%} avg")
print(f"Peak memory: {max(mem_result['memory_history_mb']):.0f} MB / {mem_result['total_kv_memory_mb']:.0f} MB")
print(f"Preemptions: {mem_result['preemptions']}")

## Part 7: Scheduling Policies

When the batch is full and new requests arrive, or when memory is tight, the scheduler
needs a policy. Common approaches:

- **FCFS (First Come, First Served)**: Simple, fair, but long requests starve short ones
- **Shortest Job First (SJF)**: Minimizes average wait time, but requires output length prediction
- **LCF (Longest Current First)**: Prioritize requests that are furthest along (reduce preemption risk)
- **Priority**: User-specified or SLA-driven priority tiers

In practice, vLLM uses FCFS with memory-based preemption (last-in-first-out for preemption victims).

In [ ]:
def compare_scheduling_policies(requests: list[Request], batch_size: int = 6) -> dict:
    """Compare FCFS vs SJF vs LCF scheduling policies."""
    import copy
    
    results = {}
    
    for policy_name, sort_key in [
        ('FCFS', lambda r: r.arrival_step),
        ('SJF (oracle)', lambda r: r.output_len),  # oracle: knows output length in advance
        ('Largest First', lambda r: -r.output_len),
    ]:
        reqs = sorted([copy.copy(r) for r in requests], key=lambda r: r.arrival_step)
        total_toks = 0
        total_steps = 0
        wait_times = []
        
        waiting = list(reqs)
        w_idx = 0
        active: list[Request] = []
        
        for current_step in range(5000):
            if w_idx >= len(waiting) and not active:
                break
            
            active = [r for r in active if not r.is_finished]
            
            # Sort candidates by policy
            candidates = [r for r in waiting[w_idx:] if r.arrival_step <= current_step]
            candidates.sort(key=sort_key)
            
            # Admit from sorted candidates
            admitted_ids = set()
            for req in candidates:
                if len(active) >= batch_size:
                    break
                req.start_step = current_step
                active.append(req)
                admitted_ids.add(req.req_id)
                wait_times.append(req.wait_time)
            
            # Advance w_idx past admitted
            while w_idx < len(waiting) and waiting[w_idx].req_id in admitted_ids:
                w_idx += 1
            
            if not active:
                if w_idx < len(waiting):
                    current_step = waiting[w_idx].arrival_step - 1
                continue
            
            total_toks += len(active)
            total_steps += 1
            for r in active:
                r.tokens_generated += 1
        
        results[policy_name] = {
            'throughput': total_toks / max(1, total_steps),
            'p50_wait': np.percentile(wait_times, 50) if wait_times else 0,
            'p99_wait': np.percentile(wait_times, 99) if wait_times else 0,
        }
    
    return results


policy_results = compare_scheduling_policies(requests[:100], batch_size=6)

print("Scheduling Policy Comparison:")
print(f"{'Policy':<20} {'Throughput':>12} {'P50 Wait':>10} {'P99 Wait':>10}")
print("-" * 55)
for policy, r in policy_results.items():
    print(f"{policy:<20} {r['throughput']:>12.2f} {r['p50_wait']:>10.0f} {r['p99_wait']:>10.0f}")

print("\nKey insight: SJF (oracle) improves P99 latency at the cost of starving long requests.")
print("FCFS is fair and simple. Production systems often use priority tiers on top of FCFS.")

## Part 8: Throughput Scaling with Batch Size

How does throughput scale as we increase max batch size? There's a sweet spot — beyond a certain
point, memory constraints prevent adding more requests, and the marginal benefit drops.

In [ ]:
batch_sizes = [1, 2, 4, 8, 16, 32]
static_throughputs = []
cont_throughputs = []

for bs in batch_sizes:
    s = simulate_static_batching(requests, batch_size=bs)
    c = simulate_continuous_batching(requests, max_batch_size=bs, max_kv_tokens=bs * 1000)
    static_throughputs.append(s.throughput)
    cont_throughputs.append(c.throughput)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(batch_sizes, static_throughputs, 'o-', color='#e74c3c', label='Static Batching', linewidth=2)
ax.plot(batch_sizes, cont_throughputs, 's-', color='#2ecc71', label='Continuous Batching', linewidth=2)

# Fill area between
ax.fill_between(batch_sizes, static_throughputs, cont_throughputs, alpha=0.15, color='#3498db',
                label='Continuous batching gain')

ax.set_xlabel("Batch Size", fontsize=12)
ax.set_ylabel("Throughput (tokens/step)", fontsize=12)
ax.set_title("Throughput Scaling with Batch Size", fontsize=13)
ax.legend(fontsize=10)
ax.set_xscale('log', base=2)
ax.set_xticks(batch_sizes)
ax.set_xticklabels(batch_sizes)
ax.grid(alpha=0.3)

# Annotate gains
for bs, s, c in zip(batch_sizes, static_throughputs, cont_throughputs):
    gain = c / s
    ax.annotate(f"{gain:.1f}x", xy=(bs, (s + c) / 2), ha='center', fontsize=8,
                color='#2980b9', fontweight='bold')

plt.tight_layout()
plt.savefig('throughput_scaling.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

Continuous batching is the single biggest throughput improvement available for LLM serving — not from better hardware or algorithms, but from better *scheduling*.

### Key Takeaways

**The Problem with Static Batching**  
Static batching blocks the GPU waiting for the longest request in a batch. With log-normal output length distributions (a few very long requests), this creates 60–80% bubble time in naive implementations.

**The Continuous Batching Solution**  
At every decode step, evict finished requests and immediately admit new ones. The batch composition changes dynamically — it's never waiting for stragglers. This turns scheduling granularity from *batch* to *token*.

**The Implementation Challenges**  
- Variable-length batch → need dynamic KV cache (PagedAttention solves this)
- Memory management → admission control based on KV token budgets  
- Preemption → swap to CPU or recompute when memory is tight
- Scheduling policy → FCFS is fair; SJF reduces average wait; priority tiers handle SLAs

**Real-World Numbers**  
vLLM's combination of continuous batching + PagedAttention achieves **10–24× higher throughput** than Hugging Face `pipeline()` on identical hardware. This is not a model improvement — it's pure systems engineering.

### What's Next (Notebook 12: Serving Metrics)

Now that we understand the serving infrastructure, we need to measure it. Notebook 12 dives into the metrics that matter in production: TTFT (time-to-first-token), TBT (time-between-tokens), throughput, and how they interact with each other under load. This is the foundation of the `benchpress` tool from the project backlog.